In [8]:

# setup cv
import os
os.environ["OPENCV_VIDEOIO_MSMF_ENABLE_HW_TRANSFORMS"] = "0" # TODO = ?
os.environ["OPENCV_VIDEOIO_MSMF_ENABLE_HW_DECODING"] = "0"
os.environ["OPENCV_VIDEOIO_MSMF_FORCE_BGR"] = "1"

# recording
from lerobot.record import RecordConfig, DatasetRecordConfig, record

# robot configs
from lerobot.cameras.opencv.configuration_opencv import OpenCVCameraConfig
from lerobot.teleoperators.so101_leader import SO101LeaderConfig, SO101Leader
from lerobot.robots.so101_follower import SO101FollowerConfig, SO101Follower

# my code
from src.paths import CALIBS_DIR, REPO_ROOT, DATASETS_DIR, HF_NAME

# set up env secrets
from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env", override=True)

True

Recording Parameters:

In [9]:
NUM_EPISODES = 5
FPS = 30
EPISODE_TIME_SEC = 60 # how long each episode
RESET_TIME_SEC   = 10 # env reset time
TASK_DESCRIPTION = 'test'
REPO_NAME = 'test1'
PUSH_TO_HUB = False

Login to HF if pushing:

In [10]:
if PUSH_TO_HUB:
    os.system(f"huggingface-cli login --token {os.getenv('HUGGINGFACE_TOKEN')}")

Robot Config:

In [11]:
camera_config = {
    "wrist_cam": OpenCVCameraConfig(index_or_path=0, width=640, height=480, fps=30),
    "top_cam": OpenCVCameraConfig(index_or_path=1, width=640, height=480, fps=30),
}

robot_config = SO101FollowerConfig(
    port="COM7",
    id="so_101_follower",
    cameras = camera_config,
    calibration_dir = CALIBS_DIR
)

teleop_config = SO101LeaderConfig(
    port="COM8",
    id="so_101_leader",
    calibration_dir = CALIBS_DIR
)
robot = SO101Follower(robot_config)
teleop = SO101Leader(teleop_config)

Dataset Config:

In [12]:
dataset_path = DATASETS_DIR / REPO_NAME
repo_id=f"{HF_NAME}/{REPO_NAME}"
root=f"{DATASETS_DIR}\\{REPO_NAME}"
resume = True if dataset_path.exists() else False

In [13]:
dscfg = DatasetRecordConfig(
    repo_id                             = repo_id,
    single_task                         = TASK_DESCRIPTION,
    root                                = root,
    fps                                 = FPS,
    episode_time_s                      = EPISODE_TIME_SEC,
    reset_time_s                        = RESET_TIME_SEC,
    num_episodes                        = NUM_EPISODES,
    video                               = True,
    push_to_hub                         = PUSH_TO_HUB,
    private                             = True,
    num_image_writer_processes          = 0,
    num_image_writer_threads_per_camera = 4,
    video_encoding_batch_size           = 1,
)

rc = RecordConfig(
    robot        = robot_config,
    dataset      = dscfg,
    teleop       = teleop_config,
    policy       = None,
    display_data = True,
    resume       = resume
)

In [14]:
record(rc, save_to_ds = False, give_feedback = False, reset_pose = None)

WARNING 2025-09-24 16:48:18 deo_utils.py:40 'torchcodec' is not available in your platform, falling back to 'pyav' as a default decoder


INFO 2025-09-24 16:48:20 a_opencv.py:182 OpenCVCamera(0) connected.
INFO 2025-09-24 16:48:21 a_opencv.py:182 OpenCVCamera(1) connected.
INFO 2025-09-24 16:48:21 follower.py:104 so_101_follower SO101Follower connected.
INFO 2025-09-24 16:48:21 01_leader.py:82 so_101_leader SO101Leader connected.
INFO 2025-09-24 16:48:21 ls\utils.py:227 Recording episode 0


Right arrow key pressed. Exiting loop...


INFO 2025-09-24 16:49:12 ls\utils.py:227 Reset the environment
Map: 100%|██████████| 1413/1413 [00:00<00:00, 1873.73 examples/s]
Generating train split: 1413 examples [00:00, 167454.55 examples/s]
Generating train split: 1 examples [00:00, 24.93 examples/s]
INFO 2025-09-24 16:49:59 ls\utils.py:227 Recording episode 1


Escape key pressed. Stopping data recording...


Map: 100%|██████████| 1111/1111 [00:00<00:00, 1957.14 examples/s]
Generating train split: 2524 examples [00:00, 350787.74 examples/s]
Generating train split: 2 examples [00:00, 92.51 examples/s]
INFO 2025-09-24 16:51:06 ls\utils.py:227 Stop recording
INFO 2025-09-24 16:51:11 a_opencv.py:488 OpenCVCamera(0) disconnected.
INFO 2025-09-24 16:51:11 a_opencv.py:488 OpenCVCamera(1) disconnected.
INFO 2025-09-24 16:51:11 follower.py:230 so_101_follower SO101Follower disconnected.
INFO 2025-09-24 16:51:11 1_leader.py:156 so_101_leader SO101Leader disconnected.
INFO 2025-09-24 16:51:11 ls\utils.py:227 Exiting


LeRobotDataset({
    Repository ID: 'jonathm126/test1',
    Number of selected episodes: '2',
    Number of selected samples: '2524',
    Features: '['action', 'observation.state', 'observation.images.wrist_cam', 'observation.images.top_cam', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',

In [15]:
# robot.disconnect()
# teleop.disconnect()